# [문제 4-1] 카페 메뉴 도구(Tool) 호출 체인 구현 (LangChain 사용)

이 노트북은 LangChain의 Tool Calling 기능을 활용하여, 다양한 데이터 소스(로컬 DB, 웹, 위키피디아)에서 정보를 검색하고 종합하여 답변하는 카페 메뉴 AI 어시스턴트를 구현합니다.

**학습 목표:**
- `@tool` 데코레이터를 사용하여 사용자 정의 도구 생성하기
- 텍스트 파일을 Chroma 벡터 DB로 구축하고 검색하기
- 서로 다른 용도의 여러 도구를 하나의 LLM에 연결(`bind_tools`)하기
- `@chain` 데코레이터를 사용하여 간단한 도구 호출 워크플로우 구현하기

In [2]:
import os
from dotenv import load_dotenv
from langchain_community.utilities.tavily_search import TAVILY_API_URL

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print(OPENAI_API_KEY[:2])
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
print(UPSTAGE_API_KEY[30:])
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
print(TAVILY_API_KEY[:2])

sk
x4
tv


### (3) 벡터 DB 생성 및 테스트 (Chroma 사용)

`cafe_menu.txt` 파일을 로드하고, 각 메뉴 항목을 별도의 `Document`로 분할한 뒤, Upstage 임베딩 모델을 사용하여 벡터로 변환합니다. 변환된 벡터는 **Chroma** 데이터베이스에 저장하여 나중에 `db_search_cafe_func` 도구가 검색할 수 있도록 준비합니다.